In [2]:
# Cell 1 — Installs & imports (run once)
# If a package install fails, re-run this cell with the appropriate pip line uncommented.
# pip install cleanlab iterstrat transformers accelerate datasets scikit-learn joblib -q


import os, sys, time, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report, f1_score, precision_score
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from cleanlab.filter import find_label_issues
from tqdm.auto import tqdm
import joblib

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available(), "device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.9.1+cu128
cuda available: True device: cuda


In [3]:
# %%
# Cell 2 — Configuration (OVERNIGHT RUN)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 128           
MODEL_PRIMARY = "roberta-base"
MODEL_SECONDARY = "roberta-base"
NUM_FOLDS = 3
FAST_DEBUG = False
N_SAMPLES_DEBUG = 512
BATCH_SIZE = 2
ACCUM_STEPS = 8

# ⭐ INCREASED FOR OVERNIGHT RUN ⭐
EPOCHS_HEAD = 2        # was 1
EPOCHS_FINETUNE = 4    # was 2
UNFREEZE_TOP_N = 4

LR_HEAD = 2e-4
LR_BASE = 1e-5
WEIGHT_DECAY = 0.01
PATIENCE = 3           # was 2
MIN_PRECISION = 0.03   
NUM_WORKERS = 2

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
print("🌙 OVERNIGHT CONFIG:", {"EPOCHS_HEAD": EPOCHS_HEAD, "EPOCHS_FINETUNE": EPOCHS_FINETUNE, "NUM_FOLDS": NUM_FOLDS})

🌙 OVERNIGHT CONFIG: {'EPOCHS_HEAD': 2, 'EPOCHS_FINETUNE': 4, 'NUM_FOLDS': 3}


In [ ]:
# %%
# CELL 3 - FINAL FIX: Load labels from CSV, ignore .npy files

DATA_DIR = Path("../data")

print("=" * 60)
print("Loading data from CSV (ignoring corrupted .npy files)")
print("=" * 60)

# Load the CSV
df_full = pd.read_csv(DATA_DIR / "processed_text.csv", keep_default_na=False)
print(f"✅ Loaded CSV with {len(df_full)} rows")

# Extract texts
text_col = "comment_text"
texts_all = df_full[text_col].astype(str).tolist()

# Extract labels DIRECTLY from CSV
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
y_all = df_full[label_cols].values.astype(np.float32)

print(f"✅ Texts: {len(texts_all)}")
print(f"✅ Labels: {y_all.shape}")

# Train / test split (using known correct sizes)
n_train = 127_656
n_test  = 31_915

texts_train = texts_all[:n_train]
texts_test = texts_all[n_train:n_train+n_test]

y_train = y_all[:n_train]
y_test = y_all[n_train:n_train+n_test]

label_names = label_cols


print("\n✅ Split complete:")
print(f"   Train: {len(texts_train)}")
print(f"   Test:  {len(texts_test)}")

# --- Alignment sanity check ---
assert len(texts_train) == len(y_train)
assert len(texts_test) == len(y_test)

print("✅ Alignment checks passed")

Loading data from CSV (ignoring corrupted .npy files)
✅ Loaded CSV with 159571 rows
✅ Texts: 159571
✅ Labels: (159571, 6)

✅ Split complete:
   Train: 127656
   Test:  31915
✅ Alignment checks passed


In [ ]:
# %%
# Cell 4 — Tokenizer, tokenization caching (FIXED - using correct data alignment)

import shutil

def get_tokenizer(model_name):
    return AutoTokenizer.from_pretrained(model_name)

tok_primary = get_tokenizer(MODEL_PRIMARY)
tok_secondary = get_tokenizer(MODEL_SECONDARY)

CACHE_DIR = Path("token_cache")

# Clear old cache (it was built with wrong label alignment)
if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
    print("🗑️  Deleted old token cache (was built with wrong labels)")

CACHE_DIR.mkdir(exist_ok=True)
print("✅ Created fresh token cache directory")

def tokenize_and_cache(name, texts, tokenizer, max_len=MAX_LEN, cache_path=None):
    cache_path = cache_path or (CACHE_DIR / f"{name}_len{max_len}.npz")
    if cache_path.exists():
        print(f"Loading token cache: {cache_path}")
        data = np.load(cache_path, allow_pickle=True)
        return {k: data[k] for k in data.files}
    
    print(f"Tokenizing and caching to: {cache_path}")
    input_ids = []
    attention_mask = []
    for i in tqdm(range(0, len(texts), 512), desc=f"Tokenizing {name}"):
        batch = texts[i:i+512]
        enc = tokenizer(batch, padding="max_length", truncation=True, max_length=max_len, return_tensors="np")
        input_ids.append(enc["input_ids"])
        attention_mask.append(enc["attention_mask"])
    
    input_ids = np.vstack(input_ids)
    attention_mask = np.vstack(attention_mask)
    np.savez_compressed(cache_path, input_ids=input_ids, attention_mask=attention_mask)
    return {"input_ids": input_ids, "attention_mask": attention_mask}

# Tokenize train + test with CORRECT alignment
# texts_train and texts_test come from the fixed Cell 3
all_texts_correct = texts_train + texts_test

print(f"\n📊 Tokenizing {len(all_texts_correct)} texts total:")
print(f"   Train: {len(texts_train)} texts")
print(f"   Test: {len(texts_test)} texts")

# Tokenize for both models
tok_cache_primary = tokenize_and_cache("primary", all_texts_correct, tok_primary)
tok_cache_secondary = tokenize_and_cache("secondary", all_texts_correct, tok_secondary)

print(f"\n✅ Tokenization complete!")
print(f"   Primary cache shape: input_ids {tok_cache_primary['input_ids'].shape}, attention_mask {tok_cache_primary['attention_mask'].shape}")
print(f"   Secondary cache shape: input_ids {tok_cache_secondary['input_ids'].shape}, attention_mask {tok_cache_secondary['attention_mask'].shape}")

# Verification: Check that cache size matches data size
expected_size = len(texts_train) + len(texts_test)
actual_size = tok_cache_primary['input_ids'].shape[0]

if expected_size == actual_size:
    print(f"✅ Cache size verification: {actual_size} == {expected_size} ✓")
else:
    print(f"❌ WARNING: Cache size mismatch! Expected {expected_size}, got {actual_size}")

🗑️  Deleted old token cache (was built with wrong labels)
✅ Created fresh token cache directory

📊 Tokenizing 159571 texts total:
   Train: 127656 texts
   Test: 31915 texts
Tokenizing and caching to: token_cache\primary_len128.npz


Tokenizing primary: 100%|██████████| 312/312 [00:35<00:00,  8.71it/s]


Tokenizing and caching to: token_cache\secondary_len128.npz


Tokenizing secondary: 100%|██████████| 312/312 [00:40<00:00,  7.64it/s]



✅ Tokenization complete!
   Primary cache shape: input_ids (159571, 128), attention_mask (159571, 128)
   Secondary cache shape: input_ids (159571, 128), attention_mask (159571, 128)
✅ Cache size verification: 159571 == 159571 ✓


In [ ]:
# %%
# Cell 5 — Dataset class (no diagnostic needed, already tested)
class TokenCacheDataset(Dataset):
    def __init__(self, token_cache, indices, labels=None):
        self.input_ids = token_cache["input_ids"]
        self.attn = token_cache["attention_mask"]
        self.indices = indices
        self.labels = labels
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        item = {
            "input_ids": torch.tensor(self.input_ids[i], dtype=torch.long),
            "attention_mask": torch.tensor(self.attn[i], dtype=torch.long)
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

print("✅ TokenCacheDataset class defined")

✅ TokenCacheDataset class defined


In [ ]:
# Cell 6 — training helpers: focal loss, optimizer builder, unfreeze helper, predict_probs
def focal_loss_with_logits(logits, targets, alpha=0.25, gamma=2.0):
    probs = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t) ** gamma * ce
    return loss.mean()

def unfreeze_top_n(model, n=4):
    # Works for transformers with encoder.layer.* structure; adapt if architecture differs
    # First freeze all
    for name, p in model.named_parameters():
        p.requires_grad = False
    # Unfreeze classifier head if present
    for name, p in model.named_parameters():
        if "classifier" in name or "pre_classifier" in name:
            p.requires_grad = True
    # Unfreeze last n encoder layers
    try:
        layers = [f"encoder.layer.{i}" for i in range(100)]
        # Deberta uses different naming; try to detect
        # heuristic: find names with "encoder.layer."
        layer_indices = sorted({int(name.split("encoder.layer.")[1].split(".")[0]) 
                                for name, _ in model.named_parameters() if "encoder.layer." in name})
        if layer_indices:
            max_idx = max(layer_indices)
            for i in range(max_idx - n + 1, max_idx + 1):
                for name, p in model.named_parameters():
                    if f"encoder.layer.{i}" in name:
                        p.requires_grad = True
    except Exception:
        pass

def build_optimizer(model, lr_head=LR_HEAD, lr_base=LR_BASE):
    # Identify head parameters by NAME, not tensor comparison
    head_names = []
    for name, p in model.named_parameters():
        if p.requires_grad and ("classifier" in name or "pre_classifier" in name):
            head_names.append(name)

    # Separate parameters using names
    head_params = [p for n, p in model.named_parameters()
                   if n in head_names and p.requires_grad]

    base_params = [p for n, p in model.named_parameters()
                   if n not in head_names and p.requires_grad]

    if len(base_params) == 0:
        print("WARNING: No base params found. Using only head params.")

    optimizer = torch.optim.AdamW(
        [
            {"params": head_params, "lr": lr_head},
            {"params": base_params, "lr": lr_base},
        ],
        weight_decay=0.01,
    )
    return optimizer

In [ ]:
# %% 
# Cell 7 — Unified predict_probs_model
@torch.no_grad()
def predict_probs_model(model, dataloader):
    """
    Predict probabilities for a dataset using a given model.
    Works for multi-label classification. Shows tqdm progress.
    """
    model.eval()
    all_probs = []

    for batch in tqdm(dataloader, desc="[predict_probs_model] test inference", leave=True):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        # mixed precision for CUDA
        with torch.amp.autocast(enabled=(DEVICE.type=="cuda"), device_type=DEVICE.type):
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())

    if all_probs:
        all_probs = np.vstack(all_probs)
    else:
        all_probs = np.zeros((0, len(label_names)))
    return all_probs


In [ ]:
# %% 
# Cell 8 — CV training with tqdm progress bars + gradient accumulation
from tqdm.auto import tqdm

def train_cv_and_predict_full(model_name, token_cache, y_full, out_model_prefix, n_folds=NUM_FOLDS):
    print("Training CV for", model_name)
    
    mskf = MultilabelStratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    folds = list(mskf.split(np.zeros(len(y_full)), y_full))
    
    fold_preds_val = np.zeros((len(y_full), y_full.shape[1]))
    fold_models = []

    for fold, (train_idx, val_idx) in enumerate(folds):
        print(f"\n=== Fold {fold+1}/{n_folds} train {len(train_idx)} val {len(val_idx)} ===")

        # --- Datasets & DataLoaders ---
        ds_train = TokenCacheDataset(token_cache, train_idx, labels=y_full[train_idx])
        ds_val   = TokenCacheDataset(token_cache, val_idx, labels=y_full[val_idx])

        # NEW CODE (simpler, no weighted sampling)
        train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        
        val_loader   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # --- Load model ---
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=y_full.shape[1]
        )
        model.to(DEVICE)

        # =========================
        # Stage 1: head-only
        # =========================
        for n, p in model.named_parameters():
            p.requires_grad = "classifier" in n or "pre_classifier" in n

        optimizer = build_optimizer(model)
        num_steps = EPOCHS_HEAD * len(train_loader) // ACCUM_STEPS
        scheduler = get_linear_schedule_with_warmup(
            optimizer, max(1, int(0.06 * num_steps)), num_steps
        )
        scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

        best_val = -1.0
        best_state = None
        stalled = 0

        for epoch in range(EPOCHS_HEAD):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loop = tqdm(train_loader, desc=f"Fold{fold+1} Head Epoch {epoch+1}", leave=False)
            running_loss = 0.0

            for batch_idx, batch in enumerate(loop):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                with torch.amp.autocast(enabled=(DEVICE.type == "cuda"), device_type=DEVICE.type):
                    logits = model(input_ids, attention_mask=attention_mask).logits
                    loss = focal_loss_with_logits(logits, labels)
                    loss = loss / ACCUM_STEPS

                scaler.scale(loss).backward()

                if (batch_idx + 1) % ACCUM_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

                running_loss += loss.item() * ACCUM_STEPS
                loop.set_postfix({"loss": running_loss / (batch_idx + 1)})

            # leftover gradients
            if (batch_idx + 1) % ACCUM_STEPS != 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            # Epoch summary
            val_probs = predict_probs_model(model, val_loader)
            val_preds = (val_probs >= optimize_thresholds(
                y_full[val_idx], val_probs, pmin=MIN_PRECISION
            )).astype(int)
            val_macro = f1_score(
                y_full[val_idx], val_preds, average="macro", zero_division=0
            )
            print(f"Fold{fold+1} Head Epoch {epoch+1} completed | Val macro F1: {val_macro:.4f}")

            if val_macro > best_val:
                best_val = val_macro
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}
                stalled = 0
            else:
                stalled += 1
                if stalled >= PATIENCE:
                    print("Early stopping head stage due to patience.")
                    break

        # =========================
        # Stage 2: finetune top-N
        # =========================
        unfreeze_top_n(model, UNFREEZE_TOP_N)
        optimizer = build_optimizer(model)
        num_steps = EPOCHS_FINETUNE * len(train_loader) // ACCUM_STEPS
        scheduler = get_linear_schedule_with_warmup(
            optimizer, max(1, int(0.06 * num_steps)), num_steps
        )
        scaler = torch.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
        stalled = 0

        for epoch in range(EPOCHS_FINETUNE):
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loop = tqdm(train_loader, desc=f"Fold{fold+1} Finetune Epoch {epoch+1}", leave=False)
            running_loss = 0.0

            for batch_idx, batch in enumerate(loop):
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                with torch.amp.autocast(enabled=(DEVICE.type == "cuda"), device_type=DEVICE.type):
                    logits = model(input_ids, attention_mask=attention_mask).logits
                    loss = focal_loss_with_logits(logits, labels)
                    loss = loss / ACCUM_STEPS

                scaler.scale(loss).backward()

                if (batch_idx + 1) % ACCUM_STEPS == 0:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    scheduler.step()

                running_loss += loss.item() * ACCUM_STEPS
                loop.set_postfix({"loss": running_loss / (batch_idx + 1)})

            if (batch_idx + 1) % ACCUM_STEPS != 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            # Epoch summary
            val_probs = predict_probs_model(model, val_loader)
            val_preds = (val_probs >= optimize_thresholds(
                y_full[val_idx], val_probs, pmin=MIN_PRECISION
            )).astype(int)
            val_macro = f1_score(
                y_full[val_idx], val_preds, average="macro", zero_division=0
            )
            print(f"Fold{fold+1} Finetune Epoch {epoch+1} completed | Val macro F1: {val_macro:.4f}")

            if val_macro > best_val:
                best_val = val_macro
                best_state = {k: v.cpu() for k, v in model.state_dict().items()}
                stalled = 0
            else:
                stalled += 1
                if stalled >= PATIENCE:
                    print("Early stopping finetune stage due to patience.")
                    break

        # --- Load best state ---
        model.load_state_dict(best_state)
        model.to(DEVICE)

        # --- Store validation predictions ---
        val_probs = predict_probs_model(model, val_loader)
        fold_preds_val[val_idx] = val_probs

        # --- Save model ---
        model_path = ARTIFACT_DIR / f"{out_model_prefix}_fold{fold}.pth"
        torch.save(model.state_dict(), model_path)
        fold_models.append(model_path)

        del model
        torch.cuda.empty_cache()

    return fold_preds_val, fold_models


In [ ]:
# %%
# Cell 8.5 — FIXED: Per-label threshold optimization

import numpy as np
from sklearn.metrics import f1_score

def optimize_thresholds(y_true, y_probs, pmin=0.03):
    """
    Optimize threshold independently for EACH label.
    Returns array of per-label thresholds.
    """
    n_labels = y_true.shape[1]
    thresholds = np.zeros(n_labels)
    
    for i in range(n_labels):
        best_f1 = -1
        best_t = 0.5
        
        # Search 100 thresholds for this specific label
        for t in np.linspace(pmin, 0.95, 100):
            preds_i = (y_probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], preds_i, zero_division=0)
            
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        
        thresholds[i] = best_t
    
    return thresholds

print("✅ Per-label threshold optimizer defined")

✅ Per-label threshold optimizer defined


In [ ]:
# %%
# Cell 9 
# 🌙 OVERNIGHT RUN with Auto-Save After Each Fold

print("="*60)
print("🌙 OVERNIGHT RUN (correct data alignment + auto-save)")
print("="*60)
print(f"Using FULL training set: {y_train.shape[0]} samples")
print(f"Config: {EPOCHS_HEAD} head + {EPOCHS_FINETUNE} finetune epochs")
print(f"Expected time: ~10-12 hours for 3 folds")
print(f"Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("="*60)

# Execute CV training with CORRECT data
fold_preds_val, fold_model_paths = train_cv_and_predict_full(
    model_name=MODEL_PRIMARY,
    token_cache=tok_cache_primary,
    y_full=y_train,  # ✅ CORRECT
    out_model_prefix="primary_v3"
)

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print(f"End time: {time.strftime('%Y-%m-%d %H:%M:%S')}")

# Save validation predictions
val_preds_path = ARTIFACT_DIR / "oof_preds_primary_v3.npy"
np.save(val_preds_path, fold_preds_val)
print(f"Saved OOF predictions to: {val_preds_path}")

# Evaluate
val_thresholds = optimize_thresholds(y_train, fold_preds_val, pmin=MIN_PRECISION)
val_preds_binary = (fold_preds_val >= val_thresholds).astype(int)
val_macro_f1 = f1_score(y_train, val_preds_binary, average='macro', zero_division=0)

print(f"\n🎯 FINAL VALIDATION MACRO F1: {val_macro_f1:.4f}")
print("\nPer-label F1:")
per_label_f1 = f1_score(y_train, val_preds_binary, average=None, zero_division=0)
for i, label in enumerate(label_names):
    print(f"  {label:15s}: {per_label_f1[i]:.4f}")

print("\n✅ Models saved as: primary_v3_fold0.pth, primary_v3_fold1.pth, primary_v3_fold2.pth")
print("\n💤 Training completed successfully!")

🌙 OVERNIGHT RUN (correct data alignment + auto-save)
Using FULL training set: 127656 samples
Config: 2 head + 4 finetune epochs
Expected time: ~10-12 hours for 3 folds
Start time: 2025-12-20 20:45:30
Training CV for roberta-base

=== Fold 1/3 train 85104 val 42552 ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[predict_probs_model] test inference: 100%|██████████| 21276/21276 [04:34<00:00, 77.48it/s]


Fold1 Head Epoch 1 completed | Val macro F1: 0.4868


[predict_probs_model] test inference: 100%|██████████| 21276/21276 [04:36<00:00, 77.06it/s]


...

Fold3 Finetune Epoch 3 completed | Val macro F1: 0.6695


[predict_probs_model] test inference: 100%|██████████| 21276/21276 [04:33<00:00, 77.74it/s]


Fold3 Finetune Epoch 4 completed | Val macro F1: 0.6747


[predict_probs_model] test inference: 100%|██████████| 21276/21276 [04:32<00:00, 77.99it/s]



✅ TRAINING COMPLETE!
End time: 2025-12-21 03:45:15
Saved OOF predictions to: artifacts\oof_preds_primary_v3.npy

🎯 FINAL VALIDATION MACRO F1: 0.6733

Per-label F1:
  toxic          : 0.8245
  severe_toxic   : 0.5223
  obscene        : 0.8318
  threat         : 0.5470
  insult         : 0.7623
  identity_hate  : 0.5517

✅ Models saved as: primary_v3_fold0.pth, primary_v3_fold1.pth, primary_v3_fold2.pth

💤 Training completed successfully!


In [ ]:
# %%
# Cell 10
# 🎯 TEST SET EVALUATION

print("="*60)
print("TEST SET EVALUATION")
print("="*60)

# Load the 3 trained models
models_primary = [
    ARTIFACT_DIR / "primary_v3_fold0.pth",
    ARTIFACT_DIR / "primary_v3_fold1.pth", 
    ARTIFACT_DIR / "primary_v3_fold2.pth"
]

# Test set indices (after training data in token cache)
test_start_idx = len(y_train)
test_end_idx = test_start_idx + len(y_test)
test_indices = np.arange(test_start_idx, test_end_idx)

print(f"Test set: indices {test_start_idx} to {test_end_idx-1}")
print(f"Test samples: {len(y_test)}")

# Create test dataset
ds_test = TokenCacheDataset(tok_cache_primary, test_indices)
test_loader = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Ensemble predictions from all 3 folds
all_test_probs = []

for i, model_path in enumerate(models_primary):
    print(f"\nLoading model {i+1}/3: {model_path.name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PRIMARY, num_labels=len(label_names)
    )
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    
    probs = predict_probs_model(model, test_loader)
    all_test_probs.append(probs)
    
    del model
    torch.cuda.empty_cache()

# Average ensemble predictions
test_probs_ensemble = np.mean(np.stack(all_test_probs, axis=0), axis=0)
print(f"\n✅ Ensemble predictions shape: {test_probs_ensemble.shape}")

# Use thresholds optimized on validation set
print("\n--- Optimizing Thresholds on Full Training Set ---")
oof_preds = np.load(ARTIFACT_DIR / "oof_preds_primary_v3.npy")
final_thresholds = optimize_thresholds(y_train, oof_preds, pmin=MIN_PRECISION)

print("\nFinal Thresholds:")
for i, label in enumerate(label_names):
    print(f"  {label:15s}: {final_thresholds[i]:.4f}")

# Apply thresholds to test set
test_preds_binary = (test_probs_ensemble >= final_thresholds).astype(int)

# Evaluate on test set
print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)

test_macro_f1 = f1_score(y_test, test_preds_binary, average='macro', zero_division=0)
print(f"\n🎯 TEST MACRO F1: {test_macro_f1:.4f}")

print("\nPer-Label Test F1:")
test_per_label_f1 = f1_score(y_test, test_preds_binary, average=None, zero_division=0)
for i, label in enumerate(label_names):
    print(f"  {label:15s}: {test_per_label_f1[i]:.4f}")

print("\nFull Classification Report:")
print(classification_report(y_test, test_preds_binary, target_names=label_names, digits=4))

# Save test predictions
test_preds_path = ARTIFACT_DIR / "test_predictions_final.npy"
np.save(test_preds_path, test_probs_ensemble)
print(f"\n✅ Saved test predictions to: {test_preds_path}")

print("\n" + "="*60)
print("🎉 EVALUATION COMPLETE!")
print("="*60)

TEST SET EVALUATION
Test set: indices 127656 to 159570
Test samples: 31915

Loading model 1/3: primary_v3_fold0.pth


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[predict_probs_model] test inference: 100%|██████████| 15958/15958 [03:17<00:00, 80.60it/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading model 2/3: primary_v3_fold1.pth


[predict_probs_model] test inference: 100%|██████████| 15958/15958 [03:51<00:00, 69.08it/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading model 3/3: primary_v3_fold2.pth


[predict_probs_model] test inference: 100%|██████████| 15958/15958 [04:01<00:00, 66.20it/s]



✅ Ensemble predictions shape: (31915, 6)

--- Optimizing Thresholds on Full Training Set ---

Final Thresholds:
  toxic          : 0.3924
  severe_toxic   : 0.3274
  obscene        : 0.3738
  threat         : 0.2995
  insult         : 0.3738
  identity_hate  : 0.3181

TEST SET RESULTS

🎯 TEST MACRO F1: 0.6769

Per-Label Test F1:
  toxic          : 0.8289
  severe_toxic   : 0.5386
  obscene        : 0.8252
  threat         : 0.5294
  insult         : 0.7725
  identity_hate  : 0.5668

Full Classification Report:
               precision    recall  f1-score   support

        toxic     0.8347    0.8232    0.8289      3037
 severe_toxic     0.4348    0.7074    0.5386       311
      obscene     0.8043    0.8472    0.8252      1669
       threat     0.4821    0.5870    0.5294        92
       insult     0.7417    0.8059    0.7725      1582
identity_hate     0.5133    0.6328    0.5668       305

    micro avg     0.7576    0.8085    0.7822      6996
    macro avg     0.6352    0.7339    0.6

c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

In [ ]:
# Cell 11 - Extra Evaluation Metrics
import numpy as np

oof_preds = np.load(ARTIFACT_DIR / "oof_preds_primary_v3.npy")
test_probs = np.load(ARTIFACT_DIR / "test_predictions_final.npy")

final_thresholds = optimize_thresholds(
    y_train,
    oof_preds,
    pmin=MIN_PRECISION
)

test_preds_binary = (test_probs >= final_thresholds).astype(int)


from sklearn.metrics import f1_score

micro_f1 = f1_score(y_test, test_preds_binary, average="micro", zero_division=0)
print(f"🎯 TEST MICRO F1: {micro_f1:.4f}")

from sklearn.metrics import precision_score, recall_score

prec = precision_score(y_test, test_preds_binary, average=None, zero_division=0)
rec  = recall_score(y_test, test_preds_binary, average=None, zero_division=0)

print("\nPer-label Precision / Recall:")
for i, label in enumerate(label_names):
    print(f"{label:15s}  P={prec[i]:.4f}  R={rec[i]:.4f}")

from sklearn.metrics import average_precision_score

print("\nPer-label PR-AUC:")
for i, label in enumerate(label_names):
    pr_auc = average_precision_score(y_test[:, i], test_probs[:, i])
    print(f"{label:15s}: {pr_auc:.4f}")




🎯 TEST MICRO F1: 0.7822

Per-label Precision / Recall:
toxic            P=0.8347  R=0.8232
severe_toxic     P=0.4348  R=0.7074
obscene          P=0.8043  R=0.8472
threat           P=0.4821  R=0.5870
insult           P=0.7417  R=0.8059
identity_hate    P=0.5133  R=0.6328

Per-label PR-AUC:
toxic          : 0.9157
severe_toxic   : 0.5481
obscene        : 0.9089
threat         : 0.5536
insult         : 0.8381
identity_hate  : 0.5615
